# Active defence — Baseline + Exhaustive Feature Search

Naive baseline (`from_Active defence` only) vs all 127 combinations of delta TQ.

In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from itertools import combinations
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Style
BG = '#FAFAFA'; GRID = '#EDEDED'; AXIS = '#D5D5D5'
TEXT = '#2D2D2D'; SUBTEXT = '#737373'
C_BASELINE = '#BF5B3F'; C_TACTICAL = '#2E74B5'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.edgecolor': AXIS, 'axes.labelcolor': SUBTEXT,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlepad': 16,
    'axes.labelsize': 9.5, 'axes.grid': False,
    'text.color': TEXT, 'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'font.family': 'sans-serif', 'figure.dpi': 140,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [10]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

QUALITY = 'Active defence'
mf["delta_Q"] = mf["to_" + QUALITY] - mf["from_" + QUALITY]

TEAM_QUALITIES = ["ATTACK", "ATTACKING_TRANSITION", "CHANCE_CREATION",
                  "DEFENCE", "DEFENSIVE_TRANSITION", "OUTCOME", "PENETRATION"]

for tq in TEAM_QUALITIES:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

delta_tq = [f"delta_tq_{tq}" for tq in TEAM_QUALITIES]
mf = mf.dropna(subset=["from_" + QUALITY, "delta_Q"] + delta_tq)

train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_Q"]
y_test  = test["delta_Q"]
print(f"Quality: {QUALITY}")
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

Quality: Active defence
Train: 3,929  |  Test: 983


---
## Naive Baseline

In [11]:
X_tr = sm.add_constant(train[["from_" + QUALITY]])
X_te = sm.add_constant(test[["from_" + QUALITY]])
naive = sm.OLS(y_train, X_tr).fit()
naive_pred = naive.predict(X_te)
naive_r2 = r2_score(y_test, naive_pred)
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive — R²: {naive_r2:.4f}  |  MAE: {naive_mae:.4f}")

Naive — R²: 0.2006  |  MAE: 0.4537


---
## Exhaustive Search (127 combinations)

In [12]:
results = []

for k in range(1, len(delta_tq) + 1):
    for combo in combinations(delta_tq, k):
        feat = ["from_" + QUALITY] + list(combo)
        X_tr = sm.add_constant(train[feat])
        X_te = sm.add_constant(test[feat])
        model = sm.OLS(y_train, X_tr).fit()
        pred = model.predict(X_te)
        results.append({
            "n_deltas": k,
            "deltas": ", ".join(c.replace("delta_tq_", "") for c in combo),
            "R2_test": r2_score(y_test, pred),
            "MAE_test": mean_absolute_error(y_test, pred),
            "R2_adj_train": model.rsquared_adj,
            "BIC": model.bic,
        })

results_df = pd.DataFrame(results).sort_values('R2_test', ascending=False)
print(f"Total combinations: {len(results_df)}")

Total combinations: 127


---
## Top 10 by Test R²

In [13]:
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(results_df.head(10).to_string(index=False, float_format="{:.4f}".format))

Naive baseline R²: 0.2006

 n_deltas                                                 deltas  R2_test  MAE_test  R2_adj_train       BIC
        2                          DEFENSIVE_TRANSITION, OUTCOME   0.2074    0.4526        0.2273 6576.4398
        3         CHANCE_CREATION, DEFENSIVE_TRANSITION, OUTCOME   0.2073    0.4526        0.2271 6584.6107
        3                  ATTACK, DEFENSIVE_TRANSITION, OUTCOME   0.2072    0.4526        0.2273 6583.5313
        4 ATTACK, CHANCE_CREATION, DEFENSIVE_TRANSITION, OUTCOME   0.2071    0.4527        0.2271 6591.7007
        1                                                OUTCOME   0.2070    0.4524        0.2272 6569.3222
        2                               CHANCE_CREATION, OUTCOME   0.2069    0.4524        0.2270 6577.5629
        2                                        ATTACK, OUTCOME   0.2068    0.4525        0.2272 6576.5681
        3                       ATTACK, CHANCE_CREATION, OUTCOME   0.2067    0.4525        0.2270 6584.8111
 

---
## Best per Number of Deltas

In [14]:
best_per_k = results_df.loc[results_df.groupby('n_deltas')['R2_test'].idxmax()]
best_per_k = best_per_k.sort_values('n_deltas')
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(best_per_k.to_string(index=False, float_format="{:.4f}".format))

Naive baseline R²: 0.2006

 n_deltas                                                                                             deltas  R2_test  MAE_test  R2_adj_train       BIC
        1                                                                                            OUTCOME   0.2070    0.4524        0.2272 6569.3222
        2                                                                      DEFENSIVE_TRANSITION, OUTCOME   0.2074    0.4526        0.2273 6576.4398
        3                                                     CHANCE_CREATION, DEFENSIVE_TRANSITION, OUTCOME   0.2073    0.4526        0.2271 6584.6107
        4                                             ATTACK, CHANCE_CREATION, DEFENSIVE_TRANSITION, OUTCOME   0.2071    0.4527        0.2271 6591.7007
        5                       ATTACK, ATTACKING_TRANSITION, CHANCE_CREATION, DEFENSIVE_TRANSITION, OUTCOME   0.2064    0.4533        0.2279 6595.1484
        6              ATTACK, ATTACKING_TRANSITION, CHANCE_C

In [15]:
best = results_df.iloc[0]
best_deltas_str = best['deltas']
best_r2 = best['R2_test']
best_mae = best['MAE_test']
print('Best combo: ' + best_deltas_str)
print('R2 test: {:.4f}  |  MAE: {:.4f}'.format(best_r2, best_mae))
print('R2 gain over naive: {:+.4f}'.format(best_r2 - naive_r2))
print()

best_deltas = ['delta_tq_' + d.strip() for d in best_deltas_str.split(',')]
best_feat = ['from_' + QUALITY] + best_deltas
X_tr = sm.add_constant(train[best_feat])
best_model = sm.OLS(y_train, X_tr).fit()
print(best_model.summary())

Best combo: DEFENSIVE_TRANSITION, OUTCOME
R2 test: 0.2074  |  MAE: 0.4526
R2 gain over naive: +0.0068

                            OLS Regression Results                            
Dep. Variable:                delta_Q   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.227
Method:                 Least Squares   F-statistic:                     386.1
Date:                Fri, 20 Mar 2026   Prob (F-statistic):          1.00e-219
Time:                        15:18:55   Log-Likelihood:                -3271.7
No. Observations:                3929   AIC:                             6551.
Df Residuals:                    3925   BIC:                             6576.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
---------